# C1-ml-fundamentals — Practice p16 — Solution

*Coding is required.*

Write code to implement

```python
def seeded_split(X, y, n_train, seed):
    ...
```

which returns the tuple `(X_train, y_train, X_test, y_test)`: a reproducible
shuffled split with the first `n_train` shuffled examples as the training
set and the rest as the test set. Contract:

- build the shuffle with `np.random.default_rng(seed).permutation(...)`;
- reorder `X` and `y` with the **same** permutation (pairs must stay glued);
- shapes: for inputs of length n, the pieces have lengths `n_train` and
  `n - n_train`.

**Banned inside the function: Python `for` and `while` loops and list
comprehensions. Any use of a banned construct scores zero points for this
problem.** Use fancy indexing and slicing, as covered in
F1-scientific-python.

Apply it to the dataset below with `n_train=15, seed=20260804`, print the
four shapes, and call it twice with the same seed to show the split is
identical both times.

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

In [ ]:
X = np.array([3.1, 7.4, 5.2, 8.8, 2.6, 6.1, 4.9, 9.3, 1.7, 5.8,
              7.9, 3.5, 6.6, 2.2, 8.1, 4.4, 9.9, 1.2, 6.9, 5.5])
y = np.array([0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0])

In [ ]:
def seeded_split(X, y, n_train, seed):
    order = np.random.default_rng(seed).permutation(len(X))
    X_shuf, y_shuf = X[order], y[order]        # one permutation, both arrays
    return (X_shuf[:n_train], y_shuf[:n_train],
            X_shuf[n_train:], y_shuf[n_train:])

X_train, y_train, X_test, y_test = seeded_split(X, y, 15, 20260804)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

# Reproducibility: same seed, same split.
again = seeded_split(X, y, 15, 20260804)
print("identical:", all(np.array_equal(a, b)
                        for a, b in zip((X_train, y_train, X_test, y_test), again)))

**Explanation.** The seeded generator makes `permutation` deterministic, so
the function is a pure recipe: same inputs, same split, every run — exactly
what reproducible experiments need. Fancy indexing with the single `order`
array reorders measurements and labels together, and two slices deal out the
train and test portions; no loop is ever required, which is why the ban is
satisfiable at all.

### Answer check

In [ ]:
assert X_train.shape == (15,) and y_train.shape == (15,)
assert X_test.shape == (5,) and y_test.shape == (5,)
assert len(np.intersect1d(X_train, X_test)) == 0
assert np.array_equal(np.sort(np.concatenate([X_train, X_test])), np.sort(X))
# pairs stayed glued: look up each test value's original label
idx = np.searchsorted(np.sort(X), X_test)
for xv, lab in zip(X_test, y_test):
    assert y[int(np.where(X == xv)[0][0])] == lab
assert np.isclose(X_test[0], 5.5, atol=1e-9, rtol=0)
import inspect
src = inspect.getsource(seeded_split)
assert "for " not in src and "while " not in src
print("p16 OK")